# GenAI ROI Analysis - Part 1: Data Exploration

**Author:** [Your Name]  
**Date:** March 2025  
**Data Source:** Stack Overflow Developer Survey 2024

---

## Project Overview

This notebook explores the Stack Overflow Developer Survey data to understand:
1. AI tool adoption patterns among developers
2. Sentiment and trust towards AI coding tools
3. Benefits and challenges of AI tool usage
4. ROI potential for organizations

---

## 1. Setup & Imports

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8-whitegrid')

# Color palette
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#A23B72',
    'success': '#28A745',
    'warning': '#FFC107',
    'danger': '#DC3545'
}

print("✅ Libraries imported successfully!")

## 2. Load Data

In [ ]:
# Load the survey data
# Update this path to your data location
DATA_PATH = '../data/raw/survey_results_public.csv'

df = pd.read_csv(DATA_PATH)
print(f"Dataset Shape: {df.shape}")
print(f"Total Respondents: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")

In [ ]:
# Display first few rows
df.head()

## 3. Identify AI-Related Columns

In [ ]:
# Find AI-related columns
ai_columns = [col for col in df.columns if 'AI' in col.upper()]
print(f"Found {len(ai_columns)} AI-related columns:")
for col in ai_columns:
    print(f"  - {col}")

In [ ]:
# Columns we'll use for analysis
COLUMNS_TO_KEEP = [
    # AI columns
    'AISelect', 'AISent', 'AIBen', 'AIAcc', 'AIComplex', 
    'AIThreat', 'AIEthics', 'AIChallenges',
    # Demographics
    'MainBranch', 'DevType', 'OrgSize', 'YearsCode', 'YearsCodePro',
    'Country', 'CompTotal', 'Currency', 'Industry', 'Employment',
    'RemoteWork', 'EdLevel', 'Age',
    # Productivity
    'TimeSearching', 'TimeAnswering', 'JobSat',
    'PurchaseInfluence', 'ICorPM', 'WorkExp'
]

# Keep only columns that exist
cols_available = [col for col in COLUMNS_TO_KEEP if col in df.columns]
print(f"Using {len(cols_available)} columns for analysis")

## 4. AI Adoption Analysis

In [ ]:
# AI Adoption Status
print("=" * 50)
print("AI TOOL ADOPTION STATUS")
print("=" * 50)

ai_adoption = df['AISelect'].value_counts()
print(ai_adoption)

# Calculate adoption rate
total = ai_adoption.sum()
adoption_rate = (ai_adoption.get('Yes', 0) / total * 100)
print(f"\n📊 Overall Adoption Rate: {adoption_rate:.1f}%")

In [ ]:
# Visualize adoption
fig, ax = plt.subplots(figsize=(8, 6))

colors = [COLORS['success'], COLORS['danger'], COLORS['warning']]
ax.pie(ai_adoption.values, labels=ai_adoption.index, autopct='%1.1f%%',
       colors=colors, explode=(0.02, 0.02, 0.02), startangle=90)
ax.set_title('AI Tool Adoption Status', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Sentiment Analysis

In [ ]:
# AI Sentiment
print("=" * 50)
print("AI TOOL SENTIMENT")
print("=" * 50)

sentiment = df['AISent'].value_counts()
print(sentiment)

In [ ]:
# Visualize sentiment
fig, ax = plt.subplots(figsize=(10, 6))

order = ['Very favorable', 'Favorable', 'Indifferent', 'Unsure', 'Unfavorable', 'Very unfavorable']
sentiment_ordered = sentiment.reindex([o for o in order if o in sentiment.index])

colors = ['#2E7D32', '#66BB6A', '#FFC107', '#29B6F6', '#FF7043', '#D32F2F']
ax.bar(range(len(sentiment_ordered)), sentiment_ordered.values, color=colors[:len(sentiment_ordered)])
ax.set_xticks(range(len(sentiment_ordered)))
ax.set_xticklabels(sentiment_ordered.index, rotation=30, ha='right')
ax.set_ylabel('Number of Respondents')
ax.set_title('AI Tool Sentiment Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Benefits Analysis

In [ ]:
# Parse benefits (multi-select field)
print("=" * 50)
print("AI TOOL BENEFITS")
print("=" * 50)

benefits = df['AIBen'].dropna().str.split(';').explode()
benefits_counts = benefits.value_counts()
print(benefits_counts.head(10))

In [ ]:
# Visualize benefits
fig, ax = plt.subplots(figsize=(10, 6))

top_benefits = benefits_counts.head(6).sort_values(ascending=True)
ax.barh(range(len(top_benefits)), top_benefits.values, color=COLORS['success'])
ax.set_yticks(range(len(top_benefits)))
ax.set_yticklabels(top_benefits.index)
ax.set_xlabel('Number of Respondents')
ax.set_title('Top Benefits of AI Coding Tools', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Challenges Analysis

In [ ]:
# Parse challenges
print("=" * 50)
print("AI TOOL CHALLENGES")
print("=" * 50)

challenges = df['AIChallenges'].dropna().str.split(';').explode()
challenges_counts = challenges.value_counts()
print(challenges_counts.head(10))

In [ ]:
# Visualize challenges
fig, ax = plt.subplots(figsize=(10, 6))

top_challenges = challenges_counts.head(6).sort_values(ascending=True)
ax.barh(range(len(top_challenges)), top_challenges.values, color=COLORS['danger'])
ax.set_yticks(range(len(top_challenges)))
ax.set_yticklabels([c[:40] + '...' if len(c) > 40 else c for c in top_challenges.index])
ax.set_xlabel('Number of Respondents')
ax.set_title('Top Challenges with AI Coding Tools', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Demographics Analysis

In [ ]:
# Company Size Distribution
print("=" * 50)
print("COMPANY SIZE DISTRIBUTION")
print("=" * 50)

print(df['OrgSize'].value_counts())

In [ ]:
# Top Countries
print("\n" + "=" * 50)
print("TOP 10 COUNTRIES")
print("=" * 50)

print(df['Country'].value_counts().head(10))

In [ ]:
# Developer Types
print("\n" + "=" * 50)
print("TOP DEVELOPER TYPES")
print("=" * 50)

print(df['DevType'].value_counts().head(10))

## 9. Adoption by Segment

In [ ]:
# Create AI user flag
df['AI_User'] = (df['AISelect'] == 'Yes').astype(int)

# Adoption by Company Size
adoption_by_size = df.groupby('OrgSize')['AI_User'].agg(['sum', 'count'])
adoption_by_size['rate'] = (adoption_by_size['sum'] / adoption_by_size['count'] * 100).round(1)
adoption_by_size = adoption_by_size.sort_values('rate', ascending=False)

print("\nAdoption Rate by Company Size:")
print(adoption_by_size)

In [ ]:
# Visualize adoption by company size
fig, ax = plt.subplots(figsize=(12, 6))

size_order = [
    'Just me - I am a freelancer, sole proprietor, etc.',
    '2 to 9 employees', '10 to 19 employees', '20 to 99 employees',
    '100 to 499 employees', '500 to 999 employees',
    '1,000 to 4,999 employees', '5,000 to 9,999 employees',
    '10,000 or more employees'
]

adoption_ordered = adoption_by_size.reindex([s for s in size_order if s in adoption_by_size.index])

ax.bar(range(len(adoption_ordered)), adoption_ordered['rate'].values, color=COLORS['primary'])
ax.set_xticks(range(len(adoption_ordered)))
ax.set_xticklabels(['Solo', '2-9', '10-19', '20-99', '100-499', '500-999', '1K-5K', '5K-10K', '10K+'],
                   rotation=45, ha='right')
ax.set_ylabel('Adoption Rate (%)')
ax.set_xlabel('Company Size (Employees)')
ax.set_title('AI Tool Adoption by Company Size', fontsize=14, fontweight='bold')
ax.axhline(y=df['AI_User'].mean()*100, color='red', linestyle='--', label='Average')
ax.legend()

plt.tight_layout()
plt.show()

## 10. Missing Values Analysis

In [ ]:
# Check missing values for our key columns
key_cols = ['AISelect', 'AISent', 'AIBen', 'AIAcc', 'AIComplex', 'AIThreat', 
            'AIChallenges', 'OrgSize', 'DevType', 'Country', 'CompTotal']

missing = df[key_cols].isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)

missing_df = pd.DataFrame({
    'Missing': missing,
    'Pct': missing_pct
}).sort_values('Pct', ascending=False)

print("Missing Values Analysis:")
print(missing_df)

## 11. Summary & Next Steps

### Key Findings from Exploration:

1. **High Adoption Rate:** ~62% of developers are using AI tools
2. **Positive Sentiment:** Majority view AI tools favorably
3. **Top Benefit:** Productivity gains
4. **Top Challenge:** Trust in AI output
5. **Company Size Effect:** Smaller companies have higher adoption

### Next Steps:
- Clean and transform data
- Calculate ROI metrics
- Create comprehensive visualizations
- Build analysis dashboard

In [ ]:
print("\n" + "=" * 50)
print("✅ Data Exploration Complete!")
print("=" * 50)
print(f"\nTotal Records: {len(df):,}")
print(f"AI Users: {df['AI_User'].sum():,} ({df['AI_User'].mean()*100:.1f}%)")
print("\nProceed to notebook 02_data_cleaning.ipynb for data preparation.")